In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/movielens-1m-with-posters-and-metadata/movies.csv
/kaggle/input/movielens-1m-with-posters-and-metadata/ratings.csv
/kaggle/input/movielens-1m-with-posters-and-metadata/users.csv
/kaggle/input/movielens-1m-dataset/users.dat
/kaggle/input/movielens-1m-dataset/ratings.dat
/kaggle/input/movielens-1m-dataset/README
/kaggle/input/movielens-1m-dataset/movies.dat


In [2]:
import os
import urllib.request
import zipfile
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

def load_movielens_1m(data_dir: str = "./data"):
    """Load MovieLens-1M dataset."""
    path = '/kaggle/input/movielens-1m-dataset/'

    # Load ratings
    ratings = pd.read_csv(
        os.path.join(path, "ratings.dat"),
        sep="::",
        names=["user_id", "movie_id", "rating", "timestamp"],
        engine="python",
        encoding="latin-1"
    )

    # Load users
    users = pd.read_csv(
        os.path.join(path, "users.dat"),
        sep="::",
        names=["user_id", "gender", "age", "occupation", "zip"],
        engine="python",
        encoding="latin-1"
    )

    # Load movies
    movies = pd.read_csv(
        os.path.join(path, "movies.dat"),
        sep="::",
        names=["movie_id", "title", "genres"],
        engine="python",
        encoding="latin-1"
    )

    return ratings, users, movies

# Load the data
ratings, users, movies = load_movielens_1m()
print(f"Ratings: {len(ratings):,}")
print(f"Users: {len(users):,}")
print(f"Movies: {len(movies):,}")

Ratings: 1,000,209
Users: 6,040
Movies: 3,883


In [3]:
def preprocess_for_implicit(
    ratings: pd.DataFrame,
    users: pd.DataFrame,
    movies: pd.DataFrame,
    min_rating: float = 4.0,  # Treat ratings >= 4 as positive
) -> dict:
    """
    Preprocess MovieLens for implicit feedback two-tower training.
    
    Returns dict with:
        - interactions: DataFrame of positive (user, item) pairs
        - user2idx: mapping from original user_id to contiguous index
        - idx2user: reverse mapping
        - item2idx: mapping from original movie_id to contiguous index
        - idx2item: reverse mapping
        - user_features: DataFrame with user features
        - item_features: DataFrame with item features
        - item_popularity: frequency of each item
    """
    # Filter to positive interactions only
    positive = ratings[ratings['rating'] >= min_rating].copy()
    print(f"Positive interactions (rating >= {min_rating}): {len(positive):,}")
    
    # Create contiguous ID mappings (required for embedding tables)
    unique_users = positive['user_id'].unique()
    unique_items = positive['movie_id'].unique()
    
    user2idx = {uid: idx for idx, uid in enumerate(unique_users)}
    idx2user = {idx: uid for uid, idx in user2idx.items()}
    item2idx = {iid: idx for idx, iid in enumerate(unique_items)}
    idx2item = {idx: iid for iid, idx in item2idx.items()}
    
    print(f"Unique users: {len(user2idx):,}")
    print(f"Unique items: {len(item2idx):,}")
    
    # Map to contiguous indices
    positive['user_idx'] = positive['user_id'].map(user2idx)
    positive['item_idx'] = positive['movie_id'].map(item2idx)
    
    # Compute item popularity (for logQ correction)
    item_counts = positive['item_idx'].value_counts()
    total_interactions = len(positive)
    item_popularity = (item_counts / total_interactions).to_dict()
    
    # Process user features
    user_features = users[users['user_id'].isin(unique_users)].copy()
    user_features['user_idx'] = user_features['user_id'].map(user2idx)
    
    # Encode categorical features
    user_features['gender_encoded'] = (user_features['gender'] == 'M').astype(int)
    
    # Age buckets: 1, 18, 25, 35, 45, 50, 56 -> normalize to 0-1
    age_map = {1: 0, 18: 0.2, 25: 0.4, 35: 0.5, 45: 0.6, 50: 0.7, 56: 0.8}
    user_features['age_normalized'] = user_features['age'].map(age_map).fillna(0.5)
    
    # Occupation: 0-20 -> one-hot or just normalize
    user_features['occupation_normalized'] = user_features['occupation'] / 20.0
    
    # Process item features (genres)
    all_genres = set()
    for genres in movies['genres']:
        all_genres.update(genres.split('|'))
    all_genres = sorted(all_genres)
    genre2idx = {g: i for i, g in enumerate(all_genres)}
    
    def encode_genres(genres_str):
        """Multi-hot encode genres."""
        vec = np.zeros(len(all_genres), dtype=np.float32)
        for g in genres_str.split('|'):
            if g in genre2idx:
                vec[genre2idx[g]] = 1.0
        return vec
    
    item_features = movies[movies['movie_id'].isin(unique_items)].copy()
    item_features['item_idx'] = item_features['movie_id'].map(item2idx)
    item_features['genre_vector'] = item_features['genres'].apply(encode_genres)
    
    return {
        'interactions': positive[['user_idx', 'item_idx', 'timestamp']],
        'user2idx': user2idx,
        'idx2user': idx2user,
        'item2idx': item2idx,
        'idx2item': idx2item,
        'user_features': user_features,
        'item_features': item_features,
        'item_popularity': item_popularity,
        'num_users': len(user2idx),
        'num_items': len(item2idx),
        'genre2idx': genre2idx,
    }

# Preprocess
data = preprocess_for_implicit(ratings, users, movies)
print(f"\nData ready for training!")
print(f"  Users: {data['num_users']:,}")
print(f"  Items: {data['num_items']:,}")
print(f"  Interactions: {len(data['interactions']):,}")

Positive interactions (rating >= 4.0): 575,281
Unique users: 6,038
Unique items: 3,533

Data ready for training!
  Users: 6,038
  Items: 3,533
  Interactions: 575,281


In [4]:
def create_temporal_split(
    interactions: pd.DataFrame,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
):
    """Split interactions temporally."""
    # Sort by timestamp
    interactions = interactions.sort_values('timestamp')

    n = len(interactions)
    train_end = int(n * (1 - val_ratio - test_ratio))
    val_end = int(n * (1 - test_ratio))

    train = interactions.iloc[:train_end]
    val = interactions.iloc[train_end:val_end]
    test = interactions.iloc[val_end:]

    return train, val, test

train_df, val_df, test_df = create_temporal_split(data['interactions'])

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader

class TwoTowerDataset(Dataset):
    """
    Dataset for two-tower model training.
    
    Each sample is a (user_idx, item_idx, item_popularity) tuple.
    We don't need explicit negatives — they come from in-batch sampling.
    """
    
    def __init__(
        self, 
        interactions: pd.DataFrame,
        item_popularity: dict[int, float],
        user_features: pd.DataFrame = None,
        item_features: pd.DataFrame = None,
    ):
        self.user_indices = torch.tensor(interactions['user_idx'].values, dtype=torch.long)
        self.item_indices = torch.tensor(interactions['item_idx'].values, dtype=torch.long)
        
        # Item popularity for logQ correction
        self.item_pop = torch.tensor(
            [item_popularity.get(i, 1e-6) for i in interactions['item_idx'].values],
            dtype=torch.float32
        )
        
        # Optional: precompute user features
        self.user_features = None
        if user_features is not None:
            # Create lookup: user_idx -> feature vector
            feature_cols = ['gender_encoded', 'age_normalized', 'occupation_normalized']
            user_feat_df = user_features.set_index('user_idx')[feature_cols]
            
            # Build tensor: [num_users, num_features]
            num_users = user_features['user_idx'].max() + 1
            self.user_feature_matrix = torch.zeros(num_users, len(feature_cols))
            for idx, row in user_feat_df.iterrows():
                self.user_feature_matrix[idx] = torch.tensor(row.values, dtype=torch.float32)
        
        # Optional: precompute item features (genre vectors)
        self.item_features = None
        if item_features is not None:
            num_items = item_features['item_idx'].max() + 1
            genre_dim = len(item_features['genre_vector'].iloc[0])
            self.item_feature_matrix = torch.zeros(num_items, genre_dim)
            for _, row in item_features.iterrows():
                self.item_feature_matrix[row['item_idx']] = torch.tensor(
                    row['genre_vector'], dtype=torch.float32
                )
    
    def __len__(self):
        return len(self.user_indices)
    
    def __getitem__(self, idx):
        user_idx = self.user_indices[idx]
        item_idx = self.item_indices[idx]
        item_pop = self.item_pop[idx]
        
        sample = {
            'user_idx': user_idx,
            'item_idx': item_idx,
            'item_pop': item_pop,
        }
        
        # Add features if available
        if self.user_feature_matrix is not None:
            sample['user_features'] = self.user_feature_matrix[user_idx]
        if self.item_feature_matrix is not None:
            sample['item_features'] = self.item_feature_matrix[item_idx]
        
        return sample

# Create datasets
train_dataset = TwoTowerDataset(
    train_df, 
    data['item_popularity'],
    data['user_features'],
    data['item_features']
)

val_dataset = TwoTowerDataset(
    val_df,
    data['item_popularity'],
    data['user_features'],
    data['item_features']
)

# Create dataloaders with large batch size for in-batch negatives
BATCH_SIZE = 1024  # Important: large batch = more negatives

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True,
    num_workers=0,  # Set > 0 for faster loading on Linux
    drop_last=True,  # Ensures consistent batch size for in-batch negatives
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

Train batches: 449
Val batches: 57


In [6]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class TwoTowerModelWithFeatures(nn.Module):
    """
    Two-tower model with optional side features.
    
    Supports:
        - User/item ID embeddings
        - Dense user features (demographics)
        - Dense item features (genre vectors)
    """
    
    def __init__(
        self,
        num_users: int,
        num_items: int,
        embedding_dim: int = 64,
        hidden_dims: list = [128, 64],
        user_feature_dim: int = 0,
        item_feature_dim: int = 0,
        dropout: float = 0.1,
    ):
        super().__init__()
        
        # ID embeddings
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        
        # Tower input dimensions
        user_input_dim = embedding_dim + user_feature_dim
        item_input_dim = embedding_dim + item_feature_dim
        
        # User tower
        self.user_tower = self._build_tower(user_input_dim, hidden_dims, dropout)
        
        # Item tower
        self.item_tower = self._build_tower(item_input_dim, hidden_dims, dropout)
        
        # Output dimension (for serving)
        self.output_dim = hidden_dims[-1]
        
        self._init_weights()
    
    def _build_tower(self, input_dim, hidden_dims, dropout):
        layers = []
        prev_dim = input_dim
        for i, hidden_dim in enumerate(hidden_dims):
            layers.append(nn.Linear(prev_dim, hidden_dim))
            layers.append(nn.ReLU())
            if i < len(hidden_dims) - 1:  # No batchnorm/dropout on last layer
                layers.append(nn.BatchNorm1d(hidden_dim))
                layers.append(nn.Dropout(dropout))
            prev_dim = hidden_dim
        return nn.Sequential(*layers)
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, mean=0, std=0.01)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
    
    def encode_user(self, user_ids, user_features=None):
        x = self.user_embedding(user_ids)
        if user_features is not None:
            x = torch.cat([x, user_features], dim=-1)
        x = self.user_tower(x)
        return F.normalize(x, dim=-1)
    
    def encode_item(self, item_ids, item_features=None):
        x = self.item_embedding(item_ids)
        if item_features is not None:
            x = torch.cat([x, item_features], dim=-1)
        x = self.item_tower(x)
        return F.normalize(x, dim=-1)
    
    def forward(self, user_ids, item_ids, user_features=None, item_features=None):
        user_emb = self.encode_user(user_ids, user_features)
        item_emb = self.encode_item(item_ids, item_features)
        return (user_emb * item_emb).sum(dim=-1)

In [7]:
def train_epoch(
    model: nn.Module,
    train_loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    temperature: float = 0.05,
    enable_log_q_correction = True,
    device: str = 'cpu',
) -> float:
    """Train for one epoch with in-batch negatives and logQ correction."""
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        user_idx = batch['user_idx'].to(device)
        item_idx = batch['item_idx'].to(device)
        item_pop = batch['item_pop'].to(device)
        
        user_features = batch.get('user_features')
        item_features = batch.get('item_features')
        if user_features is not None:
            user_features = user_features.to(device)
        if item_features is not None:
            item_features = item_features.to(device)
        
        # Forward pass: encode users and items
        user_emb = model.encode_user(user_idx, user_features)
        item_emb = model.encode_item(item_idx, item_features)
        
        # In-batch softmax with logQ correction
        logits = torch.matmul(user_emb, item_emb.T) / temperature
        
        # LogQ correction: subtract log(popularity) to debias
        if enable_log_q_correction:
            log_pop = torch.log(item_pop + 1e-10)
            logits = logits - log_pop.unsqueeze(0)
        
        # Labels: each user's positive is on the diagonal
        labels = torch.arange(logits.size(0), device=device)
        
        loss = F.cross_entropy(logits, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(train_loader)

@torch.no_grad()
def evaluate(
    model: nn.Module,
    val_loader: DataLoader,
    all_item_ids: torch.Tensor,
    item_features: torch.Tensor,
    k_values: list = [10, 50, 100],
    device: str = 'cpu',
) -> dict:
    """
    Evaluate retrieval metrics.
    
    For each user in validation, we:
    1. Encode the user
    2. Score against ALL items
    3. Compute Recall@K and NDCG@K
    """
    model.eval()
    
    # Precompute all item embeddings
    all_item_emb = model.encode_item(all_item_ids.to(device), item_features.to(device))
    
    # Collect predictions per user
    user_positives = {}  # user_idx -> set of positive item indices
    user_embeddings = {}  # user_idx -> user embedding
    
    for batch in val_loader:
        user_idx = batch['user_idx']
        item_idx = batch['item_idx']
        user_features = batch.get('user_features')
        
        if user_features is not None:
            user_features = user_features.to(device)
        
        user_emb = model.encode_user(user_idx.to(device), user_features)
        
        for i in range(len(user_idx)):
            uid = user_idx[i].item()
            iid = item_idx[i].item()
            
            if uid not in user_positives:
                user_positives[uid] = set()
                user_embeddings[uid] = user_emb[i]
            
            user_positives[uid].add(iid)
    
    # Compute metrics
    recalls = {k: [] for k in k_values}
    ndcgs = {k: [] for k in k_values}
    
    for uid, positives in user_positives.items():
        user_emb = user_embeddings[uid].unsqueeze(0)
        
        # Score all items
        scores = torch.matmul(user_emb, all_item_emb.T).squeeze(0)
        
        # Get top-K predictions
        max_k = max(k_values)
        _, top_indices = scores.topk(max_k)
        top_indices = top_indices.cpu().numpy()
        
        for k in k_values:
            top_k = set(top_indices[:k])
            
            # Recall@K: what fraction of positives are in top-K?
            hits = len(top_k & positives)
            recall = hits / min(len(positives), k)
            recalls[k].append(recall)
            
            # NDCG@K
            dcg = 0
            for i, idx in enumerate(top_indices[:k]):
                if idx in positives:
                    dcg += 1 / np.log2(i + 2)  # i+2 because i is 0-indexed
            
            # Ideal DCG: all positives at top
            idcg = sum(1 / np.log2(i + 2) for i in range(min(len(positives), k)))
            ndcg = dcg / idcg if idcg > 0 else 0
            ndcgs[k].append(ndcg)
    
    return {
        **{f'recall@{k}': np.mean(recalls[k]) for k in k_values},
        **{f'ndcg@{k}': np.mean(ndcgs[k]) for k in k_values},
    }

In [8]:
# Prepare for evaluation
all_item_ids = torch.arange(data['num_items'])
all_item_features = train_dataset.item_feature_matrix
movie_poster_df = pd.read_csv("/kaggle/input/movielens-1m-with-posters-and-metadata/movies.csv")[['movie_id', 'poster_url']]

# Evaluation

## A. Without Temperature (or Temp=1)

In [9]:
# Initialize model
USER_FEATURE_DIM = 3   # gender, age, occupation
ITEM_FEATURE_DIM = 18  # genre multi-hot vector

model = TwoTowerModelWithFeatures(
    num_users=data['num_users'],
    num_items=data['num_items'],
    embedding_dim=64,
    hidden_dims=[128, 64],
    user_feature_dim=USER_FEATURE_DIM,
    item_feature_dim=ITEM_FEATURE_DIM,
    dropout=0.1,
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

# Training loop
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)

NUM_EPOCHS = 10
TEMPERATURE = 1
enable_log_q_correction = False
print(f"Training on {device}...")
print("-" * 60)

best_recall = 0
for epoch in range(NUM_EPOCHS):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, TEMPERATURE, enable_log_q_correction, device)
    
    # Evaluate
    metrics = evaluate(
        model, val_loader, all_item_ids, all_item_features, 
        k_values=[10, 50, 100], device=device
    )
    
    # Learning rate scheduling
    scheduler.step(metrics['recall@50'])
    
    # Track best model
    if metrics['recall@50'] > best_recall:
        best_recall = metrics['recall@50']
        best_model_state = model.state_dict().copy()
    
    print(f"Epoch {epoch+1:2d} | Loss: {train_loss:.4f} | "
          f"R@10: {metrics['recall@10']:.4f} | "
          f"R@50: {metrics['recall@50']:.4f} | "
          f"R@100: {metrics['recall@100']:.4f}")

print("-" * 60)
print(f"Best Recall@50: {best_recall:.4f}")

# Load best model
model.load_state_dict(best_model_state)

Model parameters: 648,896
Training on cuda...
------------------------------------------------------------
Epoch  1 | Loss: 6.8114 | R@10: 0.0493 | R@50: 0.0554 | R@100: 0.0768
Epoch  2 | Loss: 6.7447 | R@10: 0.0597 | R@50: 0.0678 | R@100: 0.1015
Epoch  3 | Loss: 6.7331 | R@10: 0.0760 | R@50: 0.0807 | R@100: 0.1101
Epoch  4 | Loss: 6.7276 | R@10: 0.0695 | R@50: 0.0890 | R@100: 0.1225
Epoch  5 | Loss: 6.7244 | R@10: 0.0693 | R@50: 0.0839 | R@100: 0.1118
Epoch  6 | Loss: 6.7222 | R@10: 0.0834 | R@50: 0.0916 | R@100: 0.1316
Epoch  7 | Loss: 6.7206 | R@10: 0.0832 | R@50: 0.0914 | R@100: 0.1272
Epoch  8 | Loss: 6.7195 | R@10: 0.0941 | R@50: 0.0982 | R@100: 0.1392
Epoch  9 | Loss: 6.7188 | R@10: 0.0866 | R@50: 0.0951 | R@100: 0.1318
Epoch 10 | Loss: 6.7181 | R@10: 0.1002 | R@50: 0.1035 | R@100: 0.1382
------------------------------------------------------------
Best Recall@50: 0.1035


<All keys matched successfully>

## B. With Temperature (Temp=0.1)

In [10]:
# Initialize model
USER_FEATURE_DIM = 3   # gender, age, occupation
ITEM_FEATURE_DIM = 18  # genre multi-hot vector

model = TwoTowerModelWithFeatures(
    num_users=data['num_users'],
    num_items=data['num_items'],
    embedding_dim=64,
    hidden_dims=[128, 64],
    user_feature_dim=USER_FEATURE_DIM,
    item_feature_dim=ITEM_FEATURE_DIM,
    dropout=0.1,
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

# Training loop
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)

NUM_EPOCHS = 10
TEMPERATURE = 0.1
enable_log_q_correction = False
print(f"Training on {device}...")
print("-" * 60)

best_recall = 0
for epoch in range(NUM_EPOCHS):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, TEMPERATURE, enable_log_q_correction, device)
    
    # Evaluate
    metrics = evaluate(
        model, val_loader, all_item_ids, all_item_features, 
        k_values=[10, 50, 100], device=device
    )
    
    # Learning rate scheduling
    scheduler.step(metrics['recall@50'])
    
    # Track best model
    if metrics['recall@50'] > best_recall:
        best_recall = metrics['recall@50']
        best_model_state = model.state_dict().copy()
    
    print(f"Epoch {epoch+1:2d} | Loss: {train_loss:.4f} | "
          f"R@10: {metrics['recall@10']:.4f} | "
          f"R@50: {metrics['recall@50']:.4f} | "
          f"R@100: {metrics['recall@100']:.4f}")

print("-" * 60)
print(f"Best Recall@50: {best_recall:.4f}")

# Load best model
model.load_state_dict(best_model_state)

Model parameters: 648,896
Training on cuda...
------------------------------------------------------------
Epoch  1 | Loss: 6.7359 | R@10: 0.0388 | R@50: 0.0499 | R@100: 0.0762
Epoch  2 | Loss: 6.4837 | R@10: 0.0586 | R@50: 0.0836 | R@100: 0.1166
Epoch  3 | Loss: 6.3829 | R@10: 0.0788 | R@50: 0.1078 | R@100: 0.1455
Epoch  4 | Loss: 6.3245 | R@10: 0.0677 | R@50: 0.0855 | R@100: 0.1203
Epoch  5 | Loss: 6.2861 | R@10: 0.0608 | R@50: 0.0856 | R@100: 0.1315
Epoch  6 | Loss: 6.2598 | R@10: 0.0684 | R@50: 0.0922 | R@100: 0.1371
Epoch  7 | Loss: 6.2055 | R@10: 0.0757 | R@50: 0.0956 | R@100: 0.1481
Epoch  8 | Loss: 6.1930 | R@10: 0.0738 | R@50: 0.0975 | R@100: 0.1398
Epoch  9 | Loss: 6.1822 | R@10: 0.0712 | R@50: 0.0950 | R@100: 0.1428
Epoch 10 | Loss: 6.1526 | R@10: 0.0754 | R@50: 0.0958 | R@100: 0.1427
------------------------------------------------------------
Best Recall@50: 0.1078


<All keys matched successfully>

## With logQ correction with Temperature = 1

In [11]:
# Initialize model
USER_FEATURE_DIM = 3   # gender, age, occupation
ITEM_FEATURE_DIM = 18  # genre multi-hot vector

model = TwoTowerModelWithFeatures(
    num_users=data['num_users'],
    num_items=data['num_items'],
    embedding_dim=64,
    hidden_dims=[128, 64],
    user_feature_dim=USER_FEATURE_DIM,
    item_feature_dim=ITEM_FEATURE_DIM,
    dropout=0.1,
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

# Training loop
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)

NUM_EPOCHS = 10
TEMPERATURE = 1
enable_log_q_correction = True
print(f"Training on {device}...")
print("-" * 60)

best_recall = 0
for epoch in range(NUM_EPOCHS):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, TEMPERATURE, enable_log_q_correction, device)
    
    # Evaluate
    metrics = evaluate(
        model, val_loader, all_item_ids, all_item_features, 
        k_values=[10, 50, 100], device=device
    )
    
    # Learning rate scheduling
    scheduler.step(metrics['recall@50'])
    
    # Track best model
    if metrics['recall@50'] > best_recall:
        best_recall = metrics['recall@50']
        best_model_state = model.state_dict().copy()
    
    print(f"Epoch {epoch+1:2d} | Loss: {train_loss:.4f} | "
          f"R@10: {metrics['recall@10']:.4f} | "
          f"R@50: {metrics['recall@50']:.4f} | "
          f"R@100: {metrics['recall@100']:.4f}")

print("-" * 60)
print(f"Best Recall@50: {best_recall:.4f}")

# Load best model
model.load_state_dict(best_model_state)

Model parameters: 648,896
Training on cuda...
------------------------------------------------------------
Epoch  1 | Loss: 7.5054 | R@10: 0.0957 | R@50: 0.1312 | R@100: 0.1969
Epoch  2 | Loss: 7.4050 | R@10: 0.1106 | R@50: 0.1421 | R@100: 0.2209
Epoch  3 | Loss: 7.3938 | R@10: 0.1222 | R@50: 0.1581 | R@100: 0.2353
Epoch  4 | Loss: 7.3876 | R@10: 0.1251 | R@50: 0.1533 | R@100: 0.2381
Epoch  5 | Loss: 7.3827 | R@10: 0.1127 | R@50: 0.1577 | R@100: 0.2377
Epoch  6 | Loss: 7.3806 | R@10: 0.1178 | R@50: 0.1513 | R@100: 0.2272
Epoch  7 | Loss: 7.3790 | R@10: 0.1205 | R@50: 0.1582 | R@100: 0.2374
Epoch  8 | Loss: 7.3780 | R@10: 0.1333 | R@50: 0.1609 | R@100: 0.2398
Epoch  9 | Loss: 7.3779 | R@10: 0.1251 | R@50: 0.1612 | R@100: 0.2385
Epoch 10 | Loss: 7.3777 | R@10: 0.1291 | R@50: 0.1659 | R@100: 0.2442
------------------------------------------------------------
Best Recall@50: 0.1659


<All keys matched successfully>

## With logQ correction with Temperature = 0.1

In [12]:
# Initialize model
USER_FEATURE_DIM = 3   # gender, age, occupation
ITEM_FEATURE_DIM = 18  # genre multi-hot vector

model = TwoTowerModelWithFeatures(
    num_users=data['num_users'],
    num_items=data['num_items'],
    embedding_dim=64,
    hidden_dims=[128, 64],
    user_feature_dim=USER_FEATURE_DIM,
    item_feature_dim=ITEM_FEATURE_DIM,
    dropout=0.1,
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")

# Training loop
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=2
)

NUM_EPOCHS = 10
TEMPERATURE = 0.1
enable_log_q_correction = True
print(f"Training on {device}...")
print("-" * 60)

best_recall = 0
for epoch in range(NUM_EPOCHS):
    # Train
    train_loss = train_epoch(model, train_loader, optimizer, TEMPERATURE, enable_log_q_correction, device)
    
    # Evaluate
    metrics = evaluate(
        model, val_loader, all_item_ids, all_item_features, 
        k_values=[10, 50, 100], device=device
    )
    
    # Learning rate scheduling
    scheduler.step(metrics['recall@50'])
    
    # Track best model
    if metrics['recall@50'] > best_recall:
        best_recall = metrics['recall@50']
        best_model_state = model.state_dict().copy()
    
    print(f"Epoch {epoch+1:2d} | Loss: {train_loss:.4f} | "
          f"R@10: {metrics['recall@10']:.4f} | "
          f"R@50: {metrics['recall@50']:.4f} | "
          f"R@100: {metrics['recall@100']:.4f}")

print("-" * 60)
print(f"Best Recall@50: {best_recall:.4f}")

# Load best model
model.load_state_dict(best_model_state)

Model parameters: 648,896
Training on cuda...
------------------------------------------------------------
Epoch  1 | Loss: 6.8564 | R@10: 0.2212 | R@50: 0.2431 | R@100: 0.3055
Epoch  2 | Loss: 6.5023 | R@10: 0.2194 | R@50: 0.2403 | R@100: 0.3063
Epoch  3 | Loss: 6.4010 | R@10: 0.2141 | R@50: 0.2392 | R@100: 0.3084
Epoch  4 | Loss: 6.3453 | R@10: 0.2128 | R@50: 0.2392 | R@100: 0.3085
Epoch  5 | Loss: 6.2813 | R@10: 0.2114 | R@50: 0.2395 | R@100: 0.3018
Epoch  6 | Loss: 6.2613 | R@10: 0.2080 | R@50: 0.2371 | R@100: 0.3044
Epoch  7 | Loss: 6.2481 | R@10: 0.2099 | R@50: 0.2376 | R@100: 0.3027
Epoch  8 | Loss: 6.2157 | R@10: 0.2077 | R@50: 0.2384 | R@100: 0.3033
Epoch  9 | Loss: 6.2081 | R@10: 0.2084 | R@50: 0.2378 | R@100: 0.3037
Epoch 10 | Loss: 6.2028 | R@10: 0.2082 | R@50: 0.2381 | R@100: 0.3057
------------------------------------------------------------
Best Recall@50: 0.2431


<All keys matched successfully>

## Inference for User

In [13]:
import requests
from PIL import Image
from io import BytesIO
from IPython.display import display, HTML

@torch.no_grad()
def recommend_for_user(
    model: nn.Module,
    user_idx: int,
    user_features: torch.Tensor,
    all_item_emb: torch.Tensor,
    idx2item: dict,
    movies_df: pd.DataFrame,
    top_k: int = 10,
    exclude_items: set = None,
) -> pd.DataFrame:
    """
    Generate top-K recommendations for a user.
    
    Args:
        user_idx: Internal user index
        user_features: User feature vector
        all_item_emb: Precomputed item embeddings [num_items, emb_dim]
        idx2item: Mapping from index to original movie_id
        movies_df: DataFrame with movie metadata
        top_k: Number of recommendations
        exclude_items: Set of item indices to exclude (e.g., already watched)
    
    Returns:
        DataFrame with recommended movies and scores
    """
    model.eval()
    
    # Encode user
    user_emb = model.encode_user(
        torch.tensor([user_idx]).to(device),              
        user_features.unsqueeze(0).to(device, dtype=torch.float32) 
    )

    user_emb = user_emb.cpu()
    
    # Score all items
    scores = torch.matmul(user_emb, all_item_emb.T).squeeze(0)
    
    # Exclude already-watched items
    if exclude_items:
        for idx in exclude_items:
            scores[idx] = float('-inf')
    
    # Get top-K
    top_scores, top_indices = scores.topk(top_k)
    
    # Map back to movie IDs and get metadata
    recommendations = []
    for score, idx in zip(top_scores.numpy(), top_indices.numpy()):
        movie_id = idx2item[idx]
        movie_info = movies_df[movies_df['movie_id'] == movie_id].iloc[0]
        recommendations.append({
            'movie_id': movie_id,
            'title': movie_info['title'],
            'genres': movie_info['genres'],
            'score': float(score),
        })
    
    return pd.DataFrame(recommendations)

# Precompute all item embeddings for serving
model.eval()
all_item_emb = model.encode_item(
    all_item_ids.to(device),
    all_item_features.to(device)
).cpu()

# Example: Recommend for user 0
user_idx = 6037

# Get user's watch history (to exclude from recommendations)
user_history = set(train_df[train_df['user_idx'] == user_idx]['item_idx'].values)

history_movie_ids = [data['idx2item'][idx] for idx in user_history]
watched_movies = movies[movies['movie_id'].isin(history_movie_ids)].copy()
print(f"User {user_idx} Watch History (Top 12 shown):")
print("-" * 70)
movies_with_url = watched_movies[['movie_id', 'title', 'genres']].merge(movie_poster_df, how='left', on='movie_id')
html_content = '<div style="display: flex; flex-wrap: wrap; gap: 20px;">'

# Use .iterrows() to loop through the data, and .head(10) for the first 10
for index, movie in movies_with_url.head(12).iterrows():
    # Corrected the 'f' position in the img src
    html_content += f'''
        <div style="width: 150px; text-align: center;">
            <img src="{movie['poster_url']}" style="width: 100%; border-radius: 10px; box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
            <p style="font-size: 12px; margin-top: 8px; font-weight: bold;">{movie['title']}</p>
        </div>
    '''

html_content += '</div>'
from IPython.display import HTML, display
display(HTML(html_content))

# Get user features
user_feat = train_dataset.user_feature_matrix[user_idx]

print(f"\nTop 12 Recommendations:")
print("-" * 70)

recs = recommend_for_user(
    model=model,
    user_idx=user_idx,
    user_features=user_feat,
    all_item_emb=all_item_emb,
    idx2item=data['idx2item'],
    movies_df=movies,
    top_k=12,
    exclude_items=user_history,
)

recs = recs.merge(movie_poster_df, on = 'movie_id', how='left')
html_content = '<div style="display: flex; flex-wrap: wrap; gap: 20px;">'

# Use .iterrows() to loop through the data, and .head(10) for the first 10
for index, movie in recs.head(12).iterrows():
    # Corrected the 'f' position in the img src
    html_content += f'''
        <div style="width: 150px; text-align: center;">
            <img src="{movie['poster_url']}" style="width: 100%; border-radius: 10px; box-shadow: 0 4px 8px rgba(0,0,0,0.2);">
            <p style="font-size: 12px; margin-top: 8px; font-weight: bold;">{movie['title']}</p>
        </div>
    '''

html_content += '</div>'
from IPython.display import HTML, display
display(HTML(html_content))

User 6037 Watch History (Top 12 shown):
----------------------------------------------------------------------



Top 12 Recommendations:
----------------------------------------------------------------------
